# Your first ObserverBench submission

Choose a **CPU runtime**, then Run all. This trains a character-text ridge baseline from the public authorization examples, scores the labelled validation split, and makes a predictions CSV and ObserverCard. No GPU, Hugging Face token, Qwen download, or interpretability pipeline is needed. This is a starter, not a claim about the strength of text methods. Validation scores are not private-test leaderboard scores.

In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path(os.environ.get("OBSERVERBENCH_REPO", "/content/observerbench"))
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/kwisatzh/observerbench.git", str(repo)], check=True)
if not (repo / "scripts/authorization_starter.py").exists():
    raise RuntimeError("The starter update has not been published at this checkout yet.")
subprocess.run([sys.executable, "-m", "pip", "install", "numpy>=1.26,<3", "scikit-learn>=1.4,<2"], check=True)
sys.path.insert(0, str(repo / "scripts"))

## Fit on training data and get a validation score

The 768 training examples and 384 validation examples are disjoint from the 768 private-test queries. The baseline fixes its hyperparameters before validation. If you tune on these validation results, report that in your card. Larger risk means the controller should intervene sooner.

In [ ]:
from authorization_starter import run
from tempfile import mkdtemp
output = Path(mkdtemp(prefix="observerbench-")) / "submission"
report = run(output)
assert report["preflight"] == "preflight-passed"
print("Files:", output)
print("Validation loss:", report["metrics"]["protocol_loss_mean"])
print("Validation AUROC:", report["metrics"]["risk_auroc"])

## Download the submission

The ZIP contains the **test-query predictions**, ObserverCard, validation predictions, and local validation scorecard. No hidden labels are used. In your own submission, change the observer name, version, and implementation link. Attach test predictions and the card to an issue and report your validation scorecard. Opening an issue runs nothing: the maintainer validates and scores test predictions locally in batches, then reviews a draft leaderboard row.

In [ ]:
import shutil
archive = shutil.make_archive(str(output.parent / "observerbench-submission"), "zip", output)
try:
    from google.colab import files
except ImportError:
    print("Download:", archive)
else:
    files.download(archive)